## This notebook shows the convergence for a single percentile of galaxies

In [1]:
import sys
sys.path.append("/mnt/home/project/csatsangi.chetan/projects/modified_poisson")
sys.path.append("spectrophotogroup/Simulator")
sys.path.append("sahyadri-sandbox/scripts/post-process")
# import simulator
import readers
from readers import SnapshotReader
from readers import HaloReader
import numpy as np
import scipy.special as sysp 
import scipy.spatial as syspat
import pandas as pd
import gc
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from astropy.table import Table
from numpy.lib import recfunctions as rfn
from voronoi_web import Voronoi


np.int=int

In [2]:
import sys
print(sys.executable)

/mnt/home/project/csatsangi.chetan/.conda/envs/halo_env/bin/python


In [3]:
centrals = Table.read('centrals.fits').as_array()
satellites = Table.read('satellites.fits').as_array()
centrals_trimmed = rfn.drop_fields(centrals, ('Nsat','rvir'), usemask=False)
all_galaxies = np.concatenate([centrals_trimmed, satellites])

In [4]:
all_galaxies.shape

(420250,)

In [5]:
one_percentile = 0.01*all_galaxies.shape[0]
one_percentile

4202.5

In [7]:
#HaloReader parameters (Doesnt use these details for galaxies. Only accesses hr.Lbox)
real = 1
snap = 82
sim_stem = "sahyadri/default2048"
seed = 10

v = Voronoi(sim_stem=sim_stem, real=real, snap=snap, seed=seed, verbose=True, N_Proc=10)

Snapshot Reader:
... preparing to read file: /data/project/hpc2502016/data/sims/sahyadri/default2048/r1/snapdir_082/snapshot_082
... loaded header and parameters
--------------------------------
Voronoi number density estimator
--------------------------------
... using simulation sahyadri/default2048
--------------------------------
... simulation box size Lbox = 200.0 Mpc/h
... cosmology (Om,OLam,hubble) = (0.3138,0.6862,0.674)
... working at redshift z = 0.201 in realisation 1
... initialisation complete
--------------------------------


In [25]:
one_P = 0.01*all_galaxies.shape[0]
gpos = np.zeros((all_galaxies.shape[0], 3), dtype='float')
gpos[:,0] = all_galaxies['x']
gpos[:,1] = all_galaxies['y']
gpos[:,2] = all_galaxies['z']

In [20]:
Ntrc = gpos.shape[0]
percentile = 1
ran_fac_targets = np.array([4000, 10000, 20000, 40000])

results = np.zeros(len(ran_fac_targets))
for i,rf in enumerate(ran_fac_targets):
    v.Ran_Fac = rf
    delta = v.voronoi_periodic_box(gpos, ret_ran=False)
    y=1.0/(1.0+delta)
    results[i] = np.percentile(y, percentile)
    print(f"Ran_Fac={rf} done")
    
#Plot Ran_Fac/Ran_Fac_max vs VVF, per percentile
x = ran_fac_targets/ran_fac_targets.max()
plt.plot(x, results, label=f'p={percentile}%ile')
plt.xlabel(r'$Ran\_Fac/Ran\_Fac_{max}$')
plt.ylabel('$y_p$')
plt.yscale('log')
plt.legend(fontsize=8)
plt.tight_layout
plt.show()

... ... generating 4000 x Ntrc random positions
... ... generating tracer tree
... ... finding nearest nbr of each random in tracer data


KeyboardInterrupt: 

In [34]:
from scipy.spatial import cKDTree
gpos = gpos%v.Lbox
pos = gpos
Lbox = v.Lbox
Ntrc = pos.shape[0]

Ran_Fac = 3000
seed = 10

#Using voronoi_web
v.Ran_Fac = Ran_Fac

In [ ]:
delta_class = v.voronoi_periodic_box(pos, ret_ran=False)
y_class = 1.0/(1.0+delta_class)
print('Class method - mean nbr_count:',(Ran_Fac_test/(1+delta_class)).mean())

... ... generating 3000 x Ntrc random positions
... ... generating tracer tree
... ... finding nearest nbr of each random in tracer data


In [ ]:
#Using CKD Tree:
rng = np.random.RandomState(seed)
Nran = Ran_Fac * Ntrc
ran_pos = Lbox * rng.rand(Nran, 3)

tree = cKDTree(pos, boxsize=Lbox)
_, idx = tree.query(ran_pos, k=1, workers=-1)
nbr_count = np.bincount(idx, minlength=Ntrc)

delta_direct = Ran_Fac / (nbr_count + 1e-10) - 1.0
y_direct = 1.0 / (1.0 + delta_direct)